Find 10 nearest neighbors for one image

In [ ]:
import os

import matplotlib.pyplot as plt
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

EMBED_PATH = "multimodal_embeddings.pt"
QUERY_IMAGE_PATH = "real_images/apartment_4002086101_5.jpg"
TOP_K = 10
MODEL_NAME = "openai/clip-vit-base-patch32"

In [ ]:
db = torch.load(EMBED_PATH, map_location="cpu")
embeddings = db["embeddings"].float()
rows = db["rows"]

print("embeddings shape:", embeddings.shape)
print("rows:", len(rows))

if not os.path.exists(QUERY_IMAGE_PATH):
    raise FileNotFoundError(f"Image not found: {QUERY_IMAGE_PATH}")

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
processor = CLIPProcessor.from_pretrained(MODEL_NAME, use_fast=True)
model.eval()

img = Image.open(QUERY_IMAGE_PATH).convert("RGB")
inputs = processor(images=img, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    q = model.get_image_features(pixel_values=inputs["pixel_values"])
    q = q / q.norm(dim=-1, keepdim=True)

q = q.cpu()

In [ ]:
sims = (embeddings @ q.T).squeeze(1)
k = min(TOP_K, len(rows))
scores, idxs = torch.topk(sims, k=k)

results = []
for rank, (score, idx) in enumerate(zip(scores.tolist(), idxs.tolist()), start=1):
    row = rows[idx]
    image_path = os.path.join("real_images", row["filename"])
    results.append({
        "rank": rank,
        "score": score,
        "object_id": row["object_id"],
        "short_description": row["short_description"],
        "image_path": image_path,
    })

for r in results:
    print(f"#{r['rank']} | score={r['score']:.4f} | object_id={r['object_id']}")
    print(r['short_description'])
    print(r['image_path'])
    print('-' * 80)

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(10, 18))
axes = axes.ravel()

for i, ax in enumerate(axes):
    if i >= len(results):
        ax.axis('off')
        continue

    r = results[i]
    if os.path.exists(r['image_path']):
        im = Image.open(r['image_path']).convert('RGB')
        ax.imshow(im)
    ax.set_title(f"#{r['rank']} | id={r['object_id']} | {r['score']:.3f}", fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()